# Занятие 32. Практика: решающее дерево — лаборатория стабильности

Вы **пишете код и текст сами** в пустых ячейках после каждого задания. Блоки **«Легенда»** и **«Дано»** не меняйте.

Главная модель — **DecisionTreeClassifier** (решающее дерево). Теория — занятие 31, ноутбук `Урок_31_Решающее_дерево.ipynb`.

Это **не** классическое «обучи и побей accuracy». Главный вопрос практики: **насколько нервно** дерево меняет ответы, если validation слегка «подёргать» шумом. Accuracy рядом — как справочная метрика.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Split и импорты | 2 |
| 2 | Глубокое и неглубокое дерево | 4 |
| 3 | Протокол flip-rate (шум на X_val) | 5 |
| 4 | Кривая flip-rate vs сила шума | 5 |
| 5 | Какое дерево чаще меняет ответ при одном шуме | 3 |
| 6 | `plot_tree` устойчивого варианта | 3 |
| 7 | `ccp_alpha` через стабильность | 4 |
| 8 | Confusion matrix (доп. взгляд) | 2 |
| 9 | Итог: почему глубокое «нервное» | 2 |
| | **Итого** | **30** |


---
## Легенда: камера «ЖестStop» lite

В городе тестируют **учебный** прототип камеры на пешеходном переходе.
Камера смотрит на человека у края дороги и решает: **поднял ли он руку** (жест «хочу перейти») или **нет**.

Это учебный сценарий: данные **синтетические**, штрафы настоящие не выписываем.

### Признаки (таблица)

| Признак | Смысл |
|---------|--------|
| `hand_height` | насколько высоко «рука» относительно плеча |
| `arm_angle` | угол руки |
| `motion_speed` | скорость движения в кадре |
| `brightness` | яркость кадра |
| `edge_contrast` | контраст границ |
| `shadow_noise` | шум теней (часто мешает) |
| `frame_blur` | размытие кадра |
| `sensor_jitter` | дрожание сенсора (почти бесполезный шум) |

### Метка

| Значение | Смысл |
|----------|--------|
| `1` | жест «рука вверх» (хочу перейти) |
| `0` | жеста нет |

### Зачем лаборатория стабильности?

На улице признаки чуть «пляшут»: облако, блик, тряска камеры.
Если дерево из‑за мелкого шума **часто меняет ответ**, камера будет «дёргаться»: то жест есть, то нет.
Поэтому главная метрика занятия — **flip-rate** (доля объектов validation, у которых предсказание **сменилось** после шума), а не гонка за accuracy.

### Протокол (держимся его во всей практике)

1. Обучаем модели **один раз** на чистом train.
2. На **тех же** моделях берём `X_val` и добавляем **гауссов шум** разной силы (`noise_std`).
3. Считаем, какая **доля** объектов сменила предсказание относительно ответа на чистом `X_val`.
4. Модель **не** переобучаем на шуме — честно проверяем нервозность границ.

### Короткий словарь

| Слово | Значение |
|-------|----------|
| **признак** | одно число в таблице (например, `hand_height`) |
| **метка / класс** | правильный ответ: жест есть или нет |
| **train** | кадры, на которых дерево **учится** |
| **validation** | кадры для **сравнения** настроек и стабильности |
| **глубокое дерево** (`tree_deep`) | **высокое**: без потолка высоты (`max_depth=None`) — длинная цепочка вопросов |
| **неглубокое дерево** (`tree_shallow`) | **короткое**: маленький `max_depth` (эталон: 3) и/или большой `min_samples_leaf` |
| **flip-rate** | доля объектов val, у которых прогноз **изменился** после шума |
| **accuracy** | доля верных ответов (здесь — справочно) |
| **`ccp_alpha`** | сила обрезки дерева после обучения |

**Глубокое и неглубокое — как бассейн, не как «мелочь».** Глубокое дерево задаёт много вопросов подряд и режет данные на узкие кусочки. Неглубокое останавливается рано: правил меньше, поэтому лёгкий шум в признаках реже переворачивает ответ. В коде имена английские: `tree_deep` — глубокое, `tree_shallow` — неглубокое.


---
## Дано: синтетические кадры «ЖестStop»

Ячейку ниже **не меняйте**. Она создаёт таблицу признаков `X`, метки `y`, имена `FEATURE_NAMES` / `CLASS_NAMES` и фиксирует `RANDOM_STATE`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42

X, y = make_classification(
    n_samples=800,
    n_features=8,
    n_informative=3,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=2,
    flip_y=0.05,
    class_sep=1.2,
    random_state=RANDOM_STATE,
)

FEATURE_NAMES = [
    'hand_height',
    'arm_angle',
    'motion_speed',
    'brightness',
    'edge_contrast',
    'shadow_noise',
    'frame_blur',
    'sensor_jitter',
]
CLASS_NAMES = ['жеста нет', 'рука вверх']

print('Кадров:', len(X))
print('Классы [жеста нет, рука вверх]:', np.bincount(y))
pd.DataFrame(X, columns=FEATURE_NAMES).head(3)


---
## Задание 1. Split и импорты — **2 балла**

Подготовьте данные для лаборатории.

**Шаг 1.** Возьмите `RANDOM_STATE` из «Дано» (или свой — и дальше везде один и тот же): иначе нельзя честно сравнивать два дерева на одном validation.

**Шаг 2.** Разделите `X`, `y` на train и validation. Напоминание: доли классов в обеих частях должны быть похожи (это **стратификация**).
Сохраните `X_train`, `X_val`, `y_train`, `y_val`.

**Шаг 3.** Выведите размеры выборок.

Импорты уже есть в «Дано» — повторно импортировать не обязательно.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — получены `X_train`, `X_val`, `y_train`, `y_val` с фиксированным seed.
- **1.0 балл** — доли классов в train и validation близки (стратификация).

### Снижение баллов

- Нет стратификации → минус **0.5**.
- Изменён блок «Дано» → минус **1.0**.


---
## Задание 2. Глубокое и неглубокое дерево — **4 балла**

Обучите **две** модели на одном train.

**Шаг 1.** Обучите **глубокое** дерево `tree_deep` **без ограничения глубины** (`max_depth=None`) — оно может почти идеально запомнить train.

**Шаг 2.** Обучите **неглубокое** дерево `tree_shallow` с жёстким потолком высоты (в эталоне: `max_depth=3`, `min_samples_leaf=20`) — вопросов мало, правила короткие.

В коде `tree_deep` — глубокое дерево, `tree_shallow` — неглубокое. Дальше в тексте говорим по-русски.

**Шаг 3.** Для каждой модели выведите:
- глубину (`get_depth()`);
- число листьев (`get_n_leaves()`);
- accuracy на train и на validation.

**Шаг 4.** Коротко (1–2 предложения в `print` или markdown рядом): у какой модели сильнее разрыв train vs validation?

Глубина и листья — из теории занятия 31. Здесь они нужны, чтобы потом связать «сложность» со **стабильностью**.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучен `tree_deep` с `max_depth=None`.
- **1.0 балл** — обучен `tree_shallow` с `max_depth=3` и `min_samples_leaf=20`.
- **1.0 балл** — для обеих моделей напечатаны глубина, листья, train/val accuracy.
- **1.0 балл** — есть вывод про разрыв train/val (у глубокого обычно сильнее / train≈1.0).

### Снижение баллов

- Модели обучены не на `X_train` / `y_train` → минус **1.5**.
- Нет сравнения двух моделей → минус **1.0**.


---
## Задание 3. Протокол flip-rate (шум на X_val) — **5 баллов**

Реализуйте функцию стабильности.

```python
def flip_rate(model, X_val, noise_std, n_repeats=20, seed=0) -> float:
    ...
```

**Правила протокола:**

1. `base = model.predict(X_val)` — ответы на **чистом** validation.
2. Повторить `n_repeats` раз:
   - `noisy = X_val + нормальный_шум(mean=0, std=noise_std)` той же формы;
   - `pred = model.predict(noisy)`;
   - доля объектов, где `pred != base`.
3. Вернуть **среднее** этих долей (float).
4. Модель **не** трогать (`fit` внутри функции запрещён).

**Проверка:** для `noise_std=0` flip-rate должен быть **0.0** (у обеих моделей).
Посчитайте flip-rate при `noise_std=0.35` для `tree_deep` и `tree_shallow` и напечатайте.

### Подробные критерии (для проверки LLM)

- **1.5 балла** — функция сравнивает шумные предсказания с `base` на чистом `X_val`.
- **1.0 балл** — шум гауссов, форма как у `X_val`; есть усреднение по `n_repeats`.
- **1.0 балл** — при `noise_std=0` flip-rate = 0 для обеих моделей.
- **1.5 балла** — при `noise_std=0.35` flip-rate глубокого дерева **больше**, чем у неглубокого (типично ≈ в 1.5–2+ раза).

### Снижение баллов

- Переобучение модели внутри функции → минус **2.0**.
- Шум добавлен к меткам / train вместо `X_val` → минус **2.0**.
- Считается accuracy вместо доли сменившихся предсказаний → минус **1.5**.


---
## Задание 4. Кривая flip-rate vs сила шума — **5 баллов**

Постройте главную картинку лаборатории.

**Шаг 1.** Возьмите сетку сил шума, например:
`sigmas = [0.0, 0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0]`.

**Шаг 2.** Для каждого `sigma` посчитайте flip-rate у глубокого (`tree_deep`) и неглубокого (`tree_shallow`) дерева (та же функция из задания 3).

**Шаг 3.** На **одном** графике нарисуйте две линии: flip-rate vs `sigma`.
Обязательны: заголовок, подписи осей, легенда.

**Шаг 4.** Справочно напечатайте таблицу: `sigma`, `flip_deep`, `flip_shallow`, `val_acc_deep`, `val_acc_shallow`
(accuracy — на **чистом** `X_val`, она не зависит от `sigma`; это напоминание, что accuracy «стоит рядом»).

### Подробные критерии (для проверки LLM)

- **1.5 балла** — посчитаны ряды flip-rate по сетке `sigmas` для обеих моделей.
- **2.0 балла** — график line: две кривые, заголовок, оси, легенда.
- **1.5 балла** — кривая глубокого дерева лежит **выше**, чем у неглубокого, на средних/больших `sigma`; есть печать чисел.

### Снижение баллов

- Нет легенды или подписей осей → минус **1.0**.
- На графике только accuracy без flip-rate → минус **2.0**.
- Модели переобучаются на каждой силе шума → минус **1.5**.


---
## Задание 5. Какое дерево чаще меняет ответ при одном шуме — **3 балла**

Возьмите **одну** силу шума — одинаковую для обоих деревьев (одна «уличная погода» для обеих камер). Лучше середина кривой из задания 4: так разница видна, а не край «почти нет шума» / «всё сломано». В эталоне: `NOISE_COMPARE = 0.35`.

**Шаг 1.** Посчитайте flip-rate глубокого и неглубокого дерева при этом шуме.

**Шаг 2.** Нарисуйте график, на котором видно flip-rate обоих деревьев рядом (удобно двумя столбиками: по одному на дерево, высота — flip-rate). Нужны заголовок и подпись оси Y.

**Шаг 3.** Рядом напечатайте accuracy обеих моделей на чистом validation.

Смысл: даже если accuracy почти одинаковая, одно дерево может гораздо чаще менять ответ — камера «дёргается» по-разному.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — flip-rate посчитан при одной выбранной силе шума (`NOISE_COMPARE`, эталон `0.35`) для обеих моделей.
- **1.5 балла** — график, на котором видно flip-rate обоих деревьев; есть заголовок и подпись оси.
- **0.5 балла** — напечатаны справочные accuracy.

### Снижение баллов

- График без подписей / заголовка → минус **0.5**.
- Сравниваются не те модели → минус **1.0**.


---
## Задание 6. `plot_tree` устойчивого варианта — **3 балла**

Покажите, что **простые правила** читаются глазом.

**Шаг 1.** Нарисуйте `plot_tree` для `tree_shallow` (устойчивее по flip-rate).
Используйте `feature_names=FEATURE_NAMES`, `class_names=CLASS_NAMES`, `filled=True`.
Подберите `figsize`, чтобы дерево читалось.

**Шаг 2.** Одним предложением (print/markdown): почему короткое дерево обычно спокойнее реагирует на шум признаков?

### Подробные критерии (для проверки LLM)

- **2.0 балла** — построен `plot_tree` для ограниченного дерева с именами признаков и классов.
- **1.0 балл** — есть объяснение связи «мало листьев / короткие правила → устойчивее к мелкому шуму».

### Снижение баллов

- Нарисовано только глубокое дерево, неглубокое не показано → минус **1.0**.
- Нет `feature_names` / `class_names` → минус **0.5**.


---
## Задание 7. `ccp_alpha` через стабильность — **4 балла**

Обрезка дерева (`ccp_alpha`) — ещё один способ сделать правила проще.
Здесь смотрим на неё **через flip-rate**, а не через «найди лучший score».

**Шаг 1.** Для нескольких значений, например `alphas = [0.0, 0.005, 0.01, 0.02]`:
- обучите `DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE)` на train;
- запишите число листьев, validation accuracy и flip-rate при `NOISE_COMPARE`.

**Шаг 2.** Постройте график: по оси X — `ccp_alpha`, по оси Y — flip-rate (линия с маркерами).
Заголовок и подписи осей обязательны; число листьев можно напечатать таблицей рядом.

**Шаг 3.** Выберите `tree_stable` — модель с **низким** flip-rate среди разумных вариантов
(не обязательно максимальный accuracy; главное — стабильность без совсем «обрубленного» дерева из одного вопроса, если оно явно хуже по смыслу).
Кратко поясните выбор.

### Подробные критерии (для проверки LLM)

- **1.5 балла** — таблица/печать: alpha → листья, val accuracy, flip-rate.
- **1.5 балла** — график flip-rate vs `ccp_alpha` с заголовком и осями.
- **1.0 балл** — выбран `tree_stable` с явным обоснованием через стабильность.

### Снижение баллов

- Выбор только по максимальному accuracy без взгляда на flip-rate → минус **1.0**.
- Нет графика → минус **1.0**.


---
## Задание 8. Confusion matrix (доп. взгляд) — **2 балла**

Матрица ошибок — **дополнительный** взгляд: стабильность мы уже измерили flip-rate.
Здесь просто посмотрите, где ошибается устойчивая модель на чистом validation.

**Шаг 1.** Для `tree_stable` (или `tree_shallow`, если stable не выбран) постройте heatmap матрицы ошибок
(`ConfusionMatrixDisplay` или `imshow` — на ваш вкус).

**Шаг 2.** Подпишите классы через `CLASS_NAMES`. Заголовок обязателен.

**Шаг 3.** Одной фразой: confusion показывает *какие* ошибки, flip-rate — *как часто ответ прыгает* при шуме.

### Подробные критерии (для проверки LLM)

- **1.5 балла** — heatmap confusion на validation для устойчивой модели, с подписями классов.
- **0.5 балла** — есть фраза про разный смысл confusion vs flip-rate.

### Снижение баллов

- Нет заголовка / подписей → минус **0.5**.
- Считается только на train → минус **1.0**.


---
## Задание 9. Итог: почему глубокое дерево «нервное» — **2 балла**

Сформулируйте **три** коротких вывода по **своим числам** (не общие определения):

1. Что показали flip-rate глубокого и неглубокого дерева (по кривой шума и при одной выбранной силе)?
2. Как связаны глубина / число листьев / `ccp_alpha` со стабильностью?
3. Почему для камеры «ЖестStop» опаснее «нервное» дерево, даже если accuracy почти как у спокойного?

### Подробные критерии (для проверки LLM)

- **0.7 балла** — вывод с числами flip-rate глубокого и неглубокого дерева.
- **0.7 балла** — связь сложности дерева (глубина/листья/обрезка) и стабильности.
- **0.6 балла** — прикладной смысл для камеры / дрожащих предсказаний.

### Снижение баллов

- Выводы без опоры на графики/числа практики → минус **0.5**.


*(Ваш ответ)*
